# Matilda-Mini — Colab GPU validation (Phase 4, part 1: free)

Goal: confirm the code is correct on a **real CUDA GPU** before spending money on Vast.
This catches anything CPU hid (bf16 paths, fused AdamW, SDPA Flash backend).

**Note on T4:** Colab's free T4 is Turing and has *no native bf16* (it emulates, slowly).
Validate *correctness* here; do **not** trust T4 speed/MFU numbers. MFU tuning happens
in the $0.20 A100 calibration.

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## Get the code
Either `git clone` your pushed repo, or upload+unzip the `matilda-mini` folder, then `cd` in.

In [ ]:
# Option A: from GitHub (after you push)
# !git clone https://github.com/<you>/matilda-mini.git
# %cd matilda-mini

# Option B: upload a zip via the Files panel, then:
# !unzip -q matilda-mini.zip && cd matilda-mini

!pip install -q -r requirements.txt

## 1. Run the test suite on GPU
Expect **23 passed**. If anything fails here that passed on CPU, it's a real GPU-only bug.

In [ ]:
!python -m pytest tests/ -q

## 2. Tiny real-data smoke: tokenize a little FineWeb-Edu and watch loss fall
50M tokens is plenty to confirm the end-to-end path. Loss should drop from ~10.8 toward ~7 quickly.

In [ ]:
!python scripts/prepare_data.py --out-dir data/smoke --target-tokens 50000000 --shard-tokens 50000000
!python -c "import sys; sys.path.insert(0,'src'); from matilda.data import verify_manifest; print('verified:', verify_manifest('data/smoke'))"

In [ ]:
# Short real run on the smoke shard. bf16 only engages on Ampere+ (A100/L4), not T4.
!python run.py --config configs/calibration.json --data-dir data/smoke \
  --set train.total_steps=100 train.warmup_steps=10 train.batch_size=8 \
        train.compile=false train.ckpt_dir=checkpoints/smoke

## 3. Confirm resume works on GPU
Re-run the same command; it should print `[resume] from ... at step 100` and continue,
not restart. Bump `total_steps` to 150 to see it progress.

In [ ]:
!python run.py --config configs/calibration.json --data-dir data/smoke \
  --set train.total_steps=150 train.warmup_steps=10 train.batch_size=8 \
        train.compile=false train.ckpt_dir=checkpoints/smoke

If 23 tests pass, loss falls on real data, and resume continues cleanly — the code is
validated. Next: rent the A100 and run `configs/calibration.json` to tune batch size for
max MFU (with `compile=true`), then launch the long run.